# Mass Spectrometry Analysis of Extracellular Vesicles Isolated from Stimulated Human Mast Cells Exploration with `mlcroissant`

This notebook demonstrates how to use the `mlcroissant` library to load, explore, and analyze the FAIR² dataset: *Mass Spectrometry Analysis of Extracellular Vesicles Isolated from Stimulated Human Mast Cells*.

### Dataset Source
This dataset is described by a Croissant schema available at:

https://sen.science/doi/10.71728/senscience.vq4a-28xa/fair2.json

In [ ]:
# Ensure mlcroissant is installed
!pip install mlcroissant

## 1. Data Loading
Load dataset metadata and initialize the `mlcroissant` Dataset object.

In [ ]:
import mlcroissant as mlc
import pandas as pd
import pprint

# Define the Croissant schema URL
croissant_url = 'https://sen.science/doi/10.71728/senscience.vq4a-28xa/fair2.json'

# Load the dataset
dataset = mlc.Dataset(croissant_url)

# Display dataset metadata
print(f"Dataset name: {dataset.metadata.name}\n")
print(f"Dataset description: {dataset.metadata.description}\n")
print(f"Published: {dataset.metadata.datePublished}")
print(f"Version: {dataset.metadata.version}")

## 2. Data Overview
Explore available record sets, their `@id`s, and field structure using the Croissant metadata.

All entities are referenced by their Croissant `@id`, following the best practices of the schema.

In [ ]:
# List all record sets and their fields

print("Available record sets and their field @id's:")
record_sets = []
if hasattr(dataset.metadata, 'record_sets'):
    for record_set in dataset.metadata.record_sets:
        print(f"- RecordSet @id: {record_set['@id']}, name: {record_set.get('name', '')}")
        if 'fields' in record_set:
            for field in record_set['fields']:
                print(f"   - Field @id: {field['@id']}, name: {field.get('name', '')}")
        record_sets.append(record_set['@id'])
else:
    # Try dataset.metadata.record_set
    if hasattr(dataset.metadata, 'record_set'):
        for record_set in dataset.metadata.record_set:
            print(f"- RecordSet @id: {record_set['@id']}, name: {record_set.get('name', '')}")
            if 'field' in record_set:
                for field in record_set['field']:
                    print(f"   - Field @id: {field['@id']}, name: {field.get('name', '')}")
            record_sets.append(record_set['@id'])

if not record_sets:
    print("No record sets discovered in metadata. Loading one to inspect actual IDs...")
    # Sometimes metadata may not directly expose record_set(s), try inferring from .records() listing
    try:
        for rs in dataset._get_record_sets():
            print(f"- RecordSet @id: {rs['@id']}, name: {rs.get('name', '')}")
            record_sets.append(rs['@id'])
    except Exception as e:
        print(f"Failed to discover record sets from metadata: {e}")

# Show collected record sets for downstream use
print(f"\nDetected RecordSet @ids: {record_sets}")

## 3. Data Extraction
Load records from **each** record set found in the metadata, referencing them by their `@id`.

Below, data from each record set is loaded into a `DataFrame` for further processing.

In [ ]:
# Prepare dataframes for each record set
dataframes = {}
if record_sets:
    for record_set_id in record_sets:
        print(f"Loading records for RecordSet {record_set_id} ...")
        try:
            records = list(dataset.records(record_set=record_set_id))
            if (len(records) > 0) and isinstance(records[0], dict):
                df = pd.DataFrame(records)
            else:
                df = pd.DataFrame(records, columns=[str(i) for i in range(len(records[0]))]) if records else pd.DataFrame()
            dataframes[record_set_id] = df
            print(f"Fields in DataFrame for {record_set_id}: {df.columns.tolist()}")
            display(df.head())
        except Exception as e:
            print(f"Could not load records for {record_set_id}: {e}")
else:
    print("No record sets found to load.")

## 4. Exploratory Data Analysis (EDA)
Let's demonstrate filtering, normalizing, and grouping data using the loaded DataFrames.

Replace `<record_set_id>`, `<numeric_field_id>`, and `<group_field>` below with actual `@id`s visible in the printed columns above.

In [ ]:
# Pick the first non-empty dataframe
selected_record_set = None
for rs_id, df in dataframes.items():
    if not df.empty:
        selected_record_set = rs_id
        break

if selected_record_set is None:
    print("No available DataFrame found for EDA.")
else:
    print(f"Using RecordSet: {selected_record_set}")
    print(f"Columns: {dataframes[selected_record_set].columns.tolist()}")

    # Try to find a numeric field by checking dtypes
    numeric_field = None
    for col in dataframes[selected_record_set].columns:
        try:
            # Convert to numeric, drop NAs, and check if not all missing
            s = pd.to_numeric(dataframes[selected_record_set][col], errors='coerce')
            if s.notnull().sum() > 0:
                numeric_field = col
                break
        except:
            continue

    if numeric_field is None:
        print("No numeric field found for EDA.")
    else:
        print(f"Using numeric field '{numeric_field}' for analysis.")
        df_numeric = dataframes[selected_record_set].copy()
        df_numeric[numeric_field] = pd.to_numeric(df_numeric[numeric_field], errors='coerce')

        threshold = df_numeric[numeric_field].quantile(0.90)
        filtered_df = df_numeric[df_numeric[numeric_field] > threshold]
        print(f"Filtered records with {numeric_field} above 90th percentile (i.e., > {threshold}):")
        display(filtered_df.head())

        filtered_df[f"{numeric_field}_normalized"] = (filtered_df[numeric_field] - filtered_df[numeric_field].mean()) / filtered_df[numeric_field].std()
        print(f"Normalized '{numeric_field}', after filtering:")
        display(filtered_df[[numeric_field, f"{numeric_field}_normalized"]].head())

        # Attempt to group by a string/categorical field
        group_field = None
        for col in dataframes[selected_record_set].columns:
            if col != numeric_field and dataframes[selected_record_set][col].dtype == "object":
                group_field = col
                break
        if group_field:
            print(f"Grouping by field '{group_field}':")
            grouped_df = filtered_df.groupby(group_field)[numeric_field].mean()
            display(grouped_df.head())
        else:
            print("No categorical/grouping field found.")

## 5. Visualization
Let's visually explore the distribution of numeric features and group comparisons if available.
*Example below uses seaborn/matplotlib for plotting. Modify as appropriate for your application and available fields.*

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

if selected_record_set is not None and numeric_field is not None:
    plt.figure(figsize=(8,4))
    sns.histplot(dataframes[selected_record_set][numeric_field].dropna(), kde=True, bins=30)
    plt.title(f"Distribution of {numeric_field} in {selected_record_set}")
    plt.xlabel(numeric_field)
    plt.ylabel("Frequency")
    plt.show()

    if group_field:
        plt.figure(figsize=(10,4))
        sns.boxplot(x=group_field, y=numeric_field, data=dataframes[selected_record_set])
        plt.xticks(rotation=45)
        plt.title(f"{numeric_field} by {group_field}")
        plt.show()

## 6. Conclusion

In this notebook, we demonstrated how to use the `mlcroissant` Python library to explore and prepare the FAIR² dataset (referencing all data entities by their `@id` for reproducibility), load and inspect record sets, filter and normalize data, and perform basic visualizations.

Further analysis might include deeper domain-specific filtering, statistical hypothesis testing, or export of annotated datasets for downstream machine learning workflows. 